# E17 CPU harness (Kaggle pinned image)

The e17_hook unit tests and the E17 fake-server notebook harness on the same image as the GPU notebook. Kit: `foysalemonshanto/q27-e17-cpu-kit` (q27x_e17_cpu_kit_c0f30fd6b36a.bin).


In [ ]:
# E17 CPU harness on Kaggle (pinned image = the GPU notebook's): the e17_hook unit tests + the notebook harness
import hashlib, io, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_e17_cpu_kit_c0f30fd6b36a.bin", "c0f30fd6b36ab5aaca8f5a96c4e41c4c1486fb6e88cb96c8e911ab0386768450"
WH = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
OUT = Path("/kaggle/working"); K = Path("/tmp/e17_kit"); T0 = time.time()
REPORT = {"kit": KIT_NAME, "python": sys.version, "nb_sha": {}, "runs": {}}


def save():
    (OUT / "cpu_harness_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")


def find(name, sha):
    for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name, "*/*/*/*/" + name):
        for f in sorted(Path("/kaggle/input").glob(pat)):
            b = f.read_bytes()
            if hashlib.sha256(b).hexdigest() == sha:
                return b
    raise FileNotFoundError(f"{name} ({sha[:12]}) not under /kaggle/input: attach foysalemonshanto/q27-e17-cpu-kit")


shutil.rmtree(K, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(find(KIT_NAME, KIT_SHA))) as z:
    z.extractall(K)
man = json.loads((K / "MANIFEST.json").read_text())
bad = [k for k, h in man.items() if hashlib.sha256((K / k).read_bytes()).hexdigest() != h]
assert not bad, bad
for nb in sorted((K / "nb").glob("*.ipynb")):
    REPORT["nb_sha"][nb.name] = hashlib.sha256(nb.read_bytes()).hexdigest()
print("kit OK:", len(man), "files |", sys.version.split()[0], "|", REPORT["nb_sha"], flush=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "transformers==4.57.6", "huggingface-hub==0.36.2"], capture_output=True, text=True)
REPORT["pins"] = {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-800:]}
print("pins:", r.returncode, flush=True)
RUNS = [("hook_tests", [sys.executable, str(K / "run_tests.py"), str(K / "e17src"), "test_e17_hook"], K / "e17src", {}),
        ("e17_short", [sys.executable, str(K / "e17src/harness/harness_e17_nb.py"), str(K / "nb" / "Q27-XHIGH-E17_SGLANG_SILENT_ACCURACY.ipynb"), "0.03"],
         K / "e17src/harness", {"E14_HARNESS_ROOT": "/tmp/e17_hs", "HARNESS_E17_PROFILE": "E17SHORT"}),
        ("e17_full", [sys.executable, str(K / "e17src/harness/harness_e17_nb.py"), str(K / "nb" / "Q27-XHIGH-E17_SGLANG_SILENT_ACCURACY.ipynb"), "0.03"],
         K / "e17src/harness", {"E14_HARNESS_ROOT": "/tmp/e17_hf", "HARNESS_E17_PROFILE": "E17"})]
ENV = dict(os.environ, E14_SGL_SRC=str(K / "sgl_ref"), E14_FIXTURES_DIR=str(K / "fixtures"), PYTHONIOENCODING="utf-8",
           E14_HARNESS_DIR=str(K / "e14src/harness"), E15P_HARNESS_DIR=str(K / "e15p"), E17_PAYLOAD_DIR=str(K / "payload"),
           E15P_MODE="commit")
(K / "e14src/harness").mkdir(parents=True, exist_ok=True)
for name, cmd, cwd, extra in RUNS:
    t = time.time()
    try:
        r = subprocess.run(cmd, cwd=cwd, env=dict(ENV, **extra), capture_output=True, text=True, timeout=3600)
        out, rc = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        out, rc = (e.stdout or "") if isinstance(e.stdout, str) else "", "timeout"
    (OUT / f"run_{name}.log").write_text(out, encoding="utf-8")
    lines = out.splitlines()
    REPORT["runs"][name] = {"rc": rc, "sec": round(time.time() - t),
                            "ok": rc == 0 and ("CHECKS PASS" in out or " 0 failed" in out),
                            "summary": [x for x in lines if "CHECKS PASS" in x or "failed of" in x or " passed, " in x],
                            "fails": [x[:1500] for x in lines if x.startswith("FAIL")],
                            "traceback_tail": out[-3000:] if "Traceback" in out else ""}
    save()
    print(name, "->", REPORT["runs"][name]["summary"], REPORT["runs"][name]["fails"][:6], flush=True)
REPORT["sec"] = round(time.time() - T0)
REPORT["all_ok"] = all(v["ok"] for v in REPORT["runs"].values())
save()
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
